# Differential abundance analysis
This notebook will perform differential abundance analysis, using the approach generated by Andreas Fønss Møller in the hackathon

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_milo
# python -m ipykernel install --user --name scrna_cartography_milo --display-name "milo"

#### Load libraries

In [2]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# milo
import pertpy as pt
milo = pt.tl.Milo()
import mudata as md
from mudata import MuData

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np

# plotting
import matplotlib.pyplot as plt 

import random
from sklearn.metrics.pairwise import euclidean_distances
from sklearn_ann.kneighbors.annoy import AnnoyTransformer 

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import diff_genes as dg
import misc as mi
import dg_milo as dm

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Set up

In [ ]:
# Create directories
mi.create_directories(dir_path = str(here('data/milo/files')))
mi.create_directories(dir_path = str(here('data/milo/plots')))
mi.create_directories(dir_path = str(here('data/milo/objects')))

In [3]:
# Paths
base_dir = str(here('data/milo/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

In [ ]:
disease_order = ["nd", "pre", "t2d"]

#### Function to define neighborhoods, per cell type

In [4]:
from anndata import AnnData
from mudata import MuData

def make_nhoods(
    self,
    data,#: AnnData | MuData,
    neighbors_key: str | None = None,
    feature_key: str | None = "rna",
    prop: float | dict = 0.1,
    seed: int = 0,
    copy: bool = False,
    ct_key: str | None = None

):
    """Randomly sample vertices on a KNN graph to define neighbourhoods of cells.

    The set of neighborhoods get refined by computing the median profile for the neighbourhood in reduced dimensional space
    and by selecting the nearest vertex to this position.
    Thus, multiple neighbourhoods may be collapsed to prevent over-sampling the graph space.

    Args:
        data: AnnData object with KNN graph defined in `obsp` or MuData object with a modality with KNN graph defined in `obsp`
        neighbors_key: The key in `adata.obsp` or `mdata[feature_key].obsp` to use as KNN graph.
                       If not specified, `make_nhoods` looks .obsp[‘connectivities’] for connectivities (default storage places for `scanpy.pp.neighbors`).
                       If specified, it looks at .obsp[.uns[neighbors_key][‘connectivities_key’]] for connectivities.
        feature_key: If input data is MuData, specify key to cell-level AnnData object.
        prop: Fraction of cells to sample for neighbourhood index search.
        seed: Random seed for cell sampling.
        copy: Determines whether a copy of the `adata` is returned.

    Returns:
        If `copy=True`, returns the copy of `adata` with the result in `.obs`, `.obsm`, and `.uns`.
        Otherwise:

        nhoods: :class:`scipy.sparse.csr_matrix` in `adata.obsm['nhoods']`.
        A binary matrix of cell to neighbourhood assignments. Neighbourhoods in the columns are ordered by the order of the index cell in adata.obs_names

        nhood_ixs_refined: pandas.Series in `adata.obs['nhood_ixs_refined']`.
        A boolean indicating whether a cell is an index for a neighbourhood

        nhood_kth_distance: pandas.Series in `adata.obs['nhood_kth_distance']`.
        The distance to the kth nearest neighbour for each index cell (used for SpatialFDR correction)

        nhood_neighbors_key: `adata.uns["nhood_neighbors_key"]`
        KNN graph key, used for neighbourhood construction

    Examples:
        >>> import pertpy as pt
        >>> import scanpy as sc
        >>> adata = pt.dt.bhattacherjee()
        >>> milo = pt.tl.Milo()
        >>> mdata = milo.load(adata)
        >>> sc.pp.neighbors(mdata["rna"])
        >>> milo.make_nhoods(mdata["rna"])

    """
    if isinstance(data, MuData):
        adata = data[feature_key]
    if isinstance(data, AnnData):
        adata = data
    if copy:
        adata = adata.copy()

    # Get reduced dim used for KNN graph
    if neighbors_key is None:
        try:
            use_rep = adata.uns["neighbors"]["params"]["use_rep"]
        except KeyError:
            logger.warning("Using X_pca as default embedding")
            use_rep = "X_pca"
        try:
            knn_graph = adata.obsp["connectivities"].copy()
        except KeyError:
            logger.error('No "connectivities" slot in adata.obsp -- please run scanpy.pp.neighbors(adata) first')
            raise
    else:
        try:
            use_rep = adata.uns[neighbors_key]["params"]["use_rep"]
        except KeyError:
            logger.warning("Using X_pca as default embedding")
            use_rep = "X_pca"
        knn_graph = adata.obsp[neighbors_key + "_connectivities"].copy()

    X_dimred = adata.obsm[use_rep]
    
    knn_graph[knn_graph != 0] = 1
    random.seed(seed)
    
    if isinstance(prop, dict):
        vertices = []
        for key, value in prop.items():
            ct_idx = np.where(adata.obs[ct_key] == key)[0]
            ct_n_ixs = int(np.round(ct_idx.shape[0] * value))
            vertices.append(random.sample(list(ct_idx), k=ct_n_ixs))
        random_vertices = np.concat(vertices)
    else:
        n_ixs = int(np.round(adata.n_obs * prop))
        random_vertices = random.sample(range(adata.n_obs), k=n_ixs)
    
    
    random_vertices.sort()
    ixs_nn = knn_graph[random_vertices, :]
    non_zero_rows = ixs_nn.nonzero()[0]
    non_zero_cols = ixs_nn.nonzero()[1]
    refined_vertices = np.empty(
        shape=[
            len(random_vertices),
        ]
    )

    for i in range(len(random_vertices)):
        nh_pos = np.median(X_dimred[non_zero_cols[non_zero_rows == i], :], 0).reshape(-1, 1)
        nn_ixs = non_zero_cols[non_zero_rows == i]
        # Find closest real point (amongst nearest neighbors)
        dists = euclidean_distances(X_dimred[non_zero_cols[non_zero_rows == i], :], nh_pos.T)
        # Update vertex index
        refined_vertices[i] = nn_ixs[dists.argmin()]

    refined_vertices = np.unique(refined_vertices.astype("int"))
    refined_vertices.sort()

    nhoods = knn_graph[:, refined_vertices]
    adata.obsm["nhoods"] = nhoods

    # Add ixs to adata
    adata.obs["nhood_ixs_random"] = adata.obs_names.isin(adata.obs_names[random_vertices])
    adata.obs["nhood_ixs_refined"] = adata.obs_names.isin(adata.obs_names[refined_vertices])
    adata.obs["nhood_ixs_refined"] = adata.obs["nhood_ixs_refined"].astype("int")
    adata.obs["nhood_ixs_random"] = adata.obs["nhood_ixs_random"].astype("int")
    adata.uns["nhood_neighbors_key"] = neighbors_key
    # Store distance to K-th nearest neighbor (used for spatial FDR correction)
    knn_dists = adata.obsp["distances"] if neighbors_key is None else adata.obsp[neighbors_key + "_distances"]

    nhood_ixs = adata.obs["nhood_ixs_refined"] == 1
    dist_mat = knn_dists[np.asarray(nhood_ixs), :]
    k_distances = dist_mat.max(1).toarray().ravel()
    adata.obs["nhood_kth_distance"] = 0
    adata.obs["nhood_kth_distance"] = adata.obs["nhood_kth_distance"].astype(float)
    adata.obs.loc[adata.obs["nhood_ixs_refined"] == 1, "nhood_kth_distance"] = k_distances

    if copy:
        return adata

#### Load

In [ ]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

KeyboardInterrupt: 

#### Find neighbors

In [1]:
mdata = milo.load(adata)
sc.pp.neighbors(mdata["rna"], use_rep="X_latent_1", transformer=AnnoyTransformer(666))

NameError: name 'milo' is not defined

#### Create neighborhoods

In [ ]:
# Define proportion of cells
props = (1/ (mdata['rna'].obs["manual_annotation"].value_counts() / 100)).to_dict()

In [ ]:
make_nhoods(milo, mdata["rna"], prop = props, seed = 1000, ct_key='manual_annotation')

In [ ]:
mdata = milo.count_nhoods(mdata, sample_col="ic_id_platform_adjusted_sample")

In [ ]:
# Add number of cells in each sample
mdata["rna"].obs["sample_cell_count"] = (
    mdata["rna"].obs.groupby("ic_id_platform_adjusted_sample")["ic_id_platform_adjusted_sample"]
    .transform("count")
)

In [ ]:
# Annotate neighborshoods
milo.annotate_nhoods(mdata, anno_col="manual_annotation")
mdata["milo"].var["nhood_annotation_origianl"] = mdata["milo"].var["nhood_annotation"]
mdata["milo"].var["nhood_annotation"] = mdata["milo"].var["nhood_annotation"].cat.add_categories("mixed")
mdata["milo"].var.loc[mdata["milo"].var["nhood_annotation_frac"] < 0.5, "nhood_annotation"] = "mixed"

In [ ]:
import seaborn as sns
sns.distplot(mdata["milo"].var["nhood_annotation_frac"].sort_values())#.head(100)

In [ ]:
milo.build_nhood_graph(mdata)
FIGSIZE = (3, 3)
plt.rcParams["figure.figsize"] = FIGSIZE
sc.pl.embedding(adata = mdata["milo"].T, basis = "X_milo_graph", neighbors_key="nhood", color= 'nhood_annotation')
sc.pl.embedding(adata = mdata["milo"].T, basis = "X_milo_graph", neighbors_key="nhood", color= ['nhood_annotation_frac', 'Nhood_size'])

In [ ]:
# Get cells in each neighborhood
ref_cells = mdata["rna"].obs_names[mdata["rna"].obs["nhood_ixs_refined"] == 1] # Get reference cells, in neighbourhood order
nhood_to_ref = dict(enumerate(ref_cells)) # dictionary with nhood number 
nhoods = mdata["rna"].obsm["nhoods"] # Get the neighborhoods
cell_names = mdata["rna"].obs_names # Get the cell names
rows, cols = nhoods.nonzero() # numeric neighbourhood id (column index)
df = pd.DataFrame({
    "cell": cell_names[rows],
    "nhood_id": cols,                    # numeric
    "neighborhood": ref_cells[cols],     # barcode
})

In [ ]:
nhood_size = np.array(mdata["rna"].obsm["nhoods"].sum(0)).ravel()
plt.hist(nhood_size, bins=100)
plt.xlabel("# cells in nhood")
plt.ylabel("# nhoods")

#### Save

In [ ]:
df.to_csv(os.path.join(files_dir, "cells_in_neighborhood.csv"))

#### Diff abundance

In [ ]:
res_adata = dm.milo_da_nhoods(
    mdata,
    design="~sample_cell_count+disease_harmonized",
    model_contrasts="disease_harmonizedt2d-disease_harmonizednd"
)

res_adata.var.to_csv(os.path.join(files_dir, "t2d_vs_nd.csv"))

In [ ]:
res_adata = dm.milo_da_nhoods(
    mdata,
    design="~sample_cell_count+disease_harmonized",
    model_contrasts="disease_harmonizedpre-disease_harmonizednd"
)

res_adata.var.to_csv(os.path.join(files_dir, "pre_vs_nd.csv"))

#### Save mdata object

In [ ]:
cols_to_drop = ['SpatialFDR', "logCPM", "logFC", "PValue", "FDR"]
mdata["milo"].var.drop(columns=cols_to_drop, inplace=True)
mdata.write(os.path.join(objects_dir,"mdata_666_annotation.h5mu"))

#### Investigrate abundance group results

##### T2DM

In [ ]:
mdata =  md.read(os.path.join(objects_dir,"mdata_666_annotation.h5mu")) # object
res_t2d = pd.read_csv(os.path.join(files_dir, "t2d_vs_nd.csv"), index_col="index_cell") # diff abundance results
df = pd.read_csv(os.path.join(files_dir, "cells_in_neighborhood.csv")) # cells in neighborhoods

In [ ]:
filtered_df_up = res_t2d[(res_t2d["SpatialFDR"] <= 0.1) & (res_t2d["logFC"] > 0) & (res_t2d["nhood_annotation"] == "beta")]
filtered_df_down = res_t2d[(res_t2d["SpatialFDR"] <= 0.1) & (res_t2d["logFC"] < 0) & (res_t2d["nhood_annotation"] == "beta")]
cell_up=list(df[df['neighborhood'].isin(list(filtered_df_up.index))]['cell'].drop_duplicates())
cell_down=list(df[df['neighborhood'].isin(list(filtered_df_down.index))]['cell'].drop_duplicates())

In [ ]:
# Cells up and down overall
filtered_df_up = res_t2d[(res_t2d["SpatialFDR"] <= 0.1) & (res_t2d["logFC"] > 0)]
filtered_df_down = res_t2d[(res_t2d["SpatialFDR"] <= 0.1) & (res_t2d["logFC"] < 0)]
cell_up=list(df[df['neighborhood'].isin(list(filtered_df_up.index))]['cell'].drop_duplicates())
cell_down=list(df[df['neighborhood'].isin(list(filtered_df_down.index))]['cell'].drop_duplicates())

In [ ]:
# Add neighborhood annotation to df
df['index_cell'] = df['neighborhood']
df.set_index('index_cell')
df = df.merge(res_t2d[['nhood_annotation']], how = "left", on = "index_cell")

In [ ]:
# Add direction of regulation
df['direction'] = np.select(
    [
    df['index_cell'].isin(cell_up),
       df['index_cell'].isin(cell_down)
    ],
    [
        "up",
        "down"
    ],
    default="no_change"
)

In [ ]:
t2d_groups = (
    df
    .groupby(['nhood_annotation', 'direction'])['index_cell']
    .apply(list)
    .unstack(fill_value=[])
    .to_dict(orient='index')
)

#### Differentially expressed genes

In [ ]:
t2d_groups.items()

In [ ]:
# subset ann data object to neighborhoods of interest
subset = mdata['rna'][mdata['rna'].obs_names.isin(cell_up + cell_down)].copy()
# Add direction
subset.obs['direction'] = np.where(
    subset.obs.index.isin(cell_up),
    "up",
    "down" 
)
# Add  cluster cell count
subset.obs["cluster_cell_count"] = (
    subset.obs.groupby("direction")["direction"]
    .transform("count")
)

ad, dds = dg.prepare_pseudobulk_deseq_analysis(subset, 
                                  sample_key = 'ic_id_platform_adjusted_sample', 
                                  cluster_key = 'direction', 
                                n_cells = 25,
                                  design = f'~ ic_id_platform_adjusted_sample + direction', 
                                  layer="counts", 
                                  func="sum", 
                                  return_all = True, 
                                  workers=30, 
                                  no_subset=True)

In [ ]:
res = dg.diff_genes_two_clusters(dds_obj = dds, 
                              cluster_index = 'direction', 
                              cluster_1 = 'up', 
                              cluster_2 = 'down', 
                              workers = 30)
res[(res["padj"] <= 0.05) & (res["log2FoldChange"] > 0)]

In [ ]:
# Subset of cells
subset_up = mdata['rna'][cell_up]
subset_down = mdata['rna'][cell_down]
n_cells, _ = mdata['rna'].shape

In [ ]:
# Make sure you have a figure and axes
fig, axes = plt.subplots(1, 1, figsize=(5,5))  # adjust as needed

# Plot all cells in light gray
sc.pl.embedding(
    mdata['rna'], 
    basis="umap", 
    color=None, 
    palette=["grey"], 
    ax=axes, 
    show=False, 
    frameon=False, size = 3
)

axes.scatter(
    subset_down.obsm["X_umap"][:, 0],
    subset_down.obsm["X_umap"][:, 1],
    facecolors='none',
    edgecolors='blue',
    linewidths=0.8,
    s=1
)

axes.set_title("Downregulated Beta cells", fontsize=10)
plt.show()

In [ ]:
# Subset of cells
# Make sure you have a figure and axes
fig, axes = plt.subplots(1, 1, figsize=(5,5))  # adjust as needed

# Plot all cells in light gray
sc.pl.embedding(
    mdata['rna'], 
    basis="umap", 
    color=None, 
    palette=["grey"], 
    ax=axes, 
    show=False, 
    frameon=False, size = 3
)

# Overlay the subset in red
axes.scatter(
    subset_up.obsm["X_umap"][:, 0],
    subset_up.obsm["X_umap"][:, 1],
    facecolors='none',
    edgecolors='red',
    linewidths=0.8,
    s=1
)

axes.set_title("Up regulated Beta cells", fontsize=10)
plt.show()

#### Differential expressed genes¶

In [ ]:
subset = mdata['rna'][mdata['rna'].obs_names.isin(cell_up + cell_down)].copy()

subset.obs['direction'] = np.where(
    subset.obs.index.isin(cell_up),
    "up",
    "down" 
)

In [ ]:
# subset ann data object to neighborhoods of interest
subset = mdata['rna'][mdata['rna'].obs_names.isin(cell_up + cell_down)].copy()
# Add direction
subset.obs['direction'] = np.where(
    subset.obs.index.isin(cell_up),
    "up",
    "down" 
)
# Add  cluster cell count
subset.obs["cluster_cell_count"] = (
    subset.obs.groupby("direction")["direction"]
    .transform("count")
)

ad, dds = dg.prepare_pseudobulk_deseq_analysis(subset, 
                                  sample_key = 'ic_id_platform_adjusted_sample', 
                                  cluster_key = 'direction', 
                                n_cells = 25,
                                  design = f'~ ic_id_platform_adjusted_sample + direction', 
                                  layer="counts", 
                                  func="sum", 
                                  return_all = True, 
                                  workers=30, 
                                  no_subset=True)

In [ ]:
ad.groupby(['direction']).size()

In [ ]:
res = dg.diff_genes_two_clusters(dds_obj = dds, 
                              cluster_index = 'direction', 
                              cluster_1 = 'up', 
                              cluster_2 = 'down', 
                              workers = 30)
res[(res["padj"] <= 0.05) & (res["log2FoldChange"] > 0)]

In [ ]:
res[(res["padj"] <= 0.05) & (res["log2FoldChange"] > 0)].head(50)

In [ ]:
sc.pl.matrixplot(subset, list(res[(res["padj"] <= 0.05) & (res["log2FoldChange"] > 0)].head(50).index), groupby=["direction"], use_raw=False)

In [ ]:
sc.pl.matrixplot(dds, list(res[(res["padj"] <= 0.05) & (res["log2FoldChange"] > 0)].head(50).index), groupby=["direction"], layer="normed_counts")